In [ ]:
################### Cal time series #################

In [ ]:
import os
import re
import numpy as np
from nilearn.maskers import NiftiLabelsMasker
from nilearn.interfaces.fmriprep import load_confounds_strategy

def extract_number_from_filename(filename):
    match = re.search(r'(\d{3})', filename)
    return int(match.group(1)) if match else 0

data_dir = "/your/data/path"
brain_mask = "....../Brain-Mask/extendNet_24regions_updateAmy_MNI152NLin2009cAsym_res-2_space.nii.gz"

# masker
masker = NiftiLabelsMasker(
    labels_img=brain_mask,
    labels=None,
    background_label=0,
    mask_img=None,
    smoothing_fwhm=6,
    standardize=False,
    standardize_confounds=True,
    high_variance_confounds=False,
    detrend=False,
    low_pass=0.08,
    high_pass=0.01,
    t_r=1.5,
    dtype=None,
    resampling_target="labels",
    memory_level=1,
    verbose=0,
    strategy='mean',
    keep_masked_labels=True,
    reports=True
)

time_series_list_preXXXRest = []  # XXX-Task
subject_ids = []

for subj_folder in sorted(os.listdir(data_dir), key=extract_number_from_filename):
    subj_path = os.path.join(data_dir, subj_folder, 'func')
    if os.path.isdir(subj_path):

        for file in sorted(os.listdir(subj_path), key=extract_number_from_filename):
            if file.endswith('task-preXXXRest_space-MNI152NLin2009cAsym_res-2_desc-preproc_bold.nii.gz'):
                fmri_img = os.path.join(subj_path, file)

                subject_id = subj_folder

                confounds, sample_mask = load_confounds_strategy(
                    fmri_img,
                    denoise_strategy="scrubbing",
                    fd_threshold=0.5,
                    std_dvars_threshold=3,
                    demean=True
                )

                time_series = masker.fit_transform(
                    fmri_img,
                    confounds=confounds,
                    sample_mask=sample_mask
                )

                time_series_list_preRecallRest.append(time_series)
                subject_ids.append(subject_id)

print(f"Number of subjects: {len(time_series_list_preRecallRest)}")

arr_obj = np.array(time_series_list_preRecallRest, dtype=object)
np.save("time_series_list_preXXXRest.npy", arr_obj, allow_pickle=True)

for i, time_series in enumerate(time_series_list_preRecallRest):
    print(f"Subject {i+1} ({subject_ids[i]}) time series shape: {time_series.shape}")